In [1]:
pip install tensorflow numpy matplotlib


Defaulting to user installation because normal site-packages is not writeable
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl.metadata (46 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 1.2 MB/s eta 0:04:49
   ---------------------------------------- 1.0/351.2 MB 1.4 MB/s eta 0:04:12
   ---------------------------------------- 1.3/351.2 MB 1.5 MB/s eta 0:03:57
   ---------------------------------------- 1.8/351.2 MB 1.6 MB/s eta 0:03:33
   ---------------------------------------- 2.4/351.2 MB 1.7 MB/s eta 0:03:20
   ---------------------------------------- 2.9/351.2 MB 1.9 MB/s eta 0:03:07
   -------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\rchalla\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
import pickle

# -----------------------------
# Read story
# -----------------------------

with open("story.txt", "r", encoding="utf-8") as file:
    text = file.read().lower()

# -----------------------------
# Tokenization
# -----------------------------

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)

# -----------------------------
# Create sequences
# -----------------------------

input_sequences = []

for line in text.split("\n"):

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        sequence = token_list[:i + 1]
        input_sequences.append(sequence)

# -----------------------------
# Padding
# -----------------------------

max_sequence_length = max(
    len(sequence) for sequence in input_sequences
)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_length,
        padding="pre"
    )
)

# -----------------------------
# X and y
# -----------------------------

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = tf.keras.utils.to_categorical(
    y,
    num_classes=total_words
)

# -----------------------------
# Build RNN
# -----------------------------

model = Sequential()

model.add(
    Embedding(
        total_words,
        50,
        input_length=max_sequence_length - 1
    )
)

model.add(SimpleRNN(100))

model.add(
    Dense(
        total_words,
        activation="softmax"
    )
)

# -----------------------------
# Compile
# -----------------------------

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# -----------------------------
# Train
# -----------------------------

print("\nTraining model...")

model.fit(
    X,
    y,
    epochs=100,
    batch_size=32
)

print("\nTraining completed!")

# -----------------------------
# Save model
# -----------------------------

model.save("next_word_rnn.keras")

# Save tokenizer
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

# Save sequence length
with open("sequence_length.pkl", "wb") as file:
    pickle.dump(max_sequence_length, file)

print("\nModel saved successfully!")


Vocabulary size: 240

Training model...
Epoch 1/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.0646 - loss: 5.3112
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1036 - loss: 4.8909
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1036 - loss: 4.8181
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1036 - loss: 4.7895
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1036 - loss: 4.7731
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1036 - loss: 4.7289
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1036 - loss: 4.6772
Epoch 8/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1096 - loss: 4.5797
Epoch 9/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1381 - loss: 4.4491
Epoch 10/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1697 - loss: 4.2852
Epoch 11/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1982 - loss: 4.0935
Epoch 12/100
